# Baseline: logistic regression without rebalancing

Goal: a first reference model, trained on the imbalanced data

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from src.config import SEED
from src.data import load_splits
from src.metrics import evaluate, confusion_rows

sns.set_theme(style='whitegrid')

data, y_train, y_test = load_splits()
data['salary']['X_train_s'].shape, data['no_salary']['X_train_s'].shape

((51416, 26), (51416, 25))

## always predict the most frequent class

In [2]:
dummy = DummyClassifier(strategy='most_frequent').fit(data['no_salary']['X_train_s'], y_train)
summary, _ = evaluate(y_test, dummy.predict(data['no_salary']['X_test_s']))
summary

{'accuracy': 0.31116297160637885, 'macro_f1': 0.09492732126965292}

## Logistic regression on both feature sets

Trained on the scaled data, no class weights, no resampling.

In [3]:
results, per_class, preds = {}, {}, {}
for name, d in data.items():
    model = LogisticRegression(max_iter=1000, random_state=SEED)
    model.fit(d['X_train_s'], y_train)
    preds[name] = model.predict(d['X_test_s'])
    results[name], per_class[name] = evaluate(y_test, preds[name])

pd.DataFrame(results).T.round(4)

,accuracy,macro_f1
salary,0.3784,0.2153
no_salary,0.3140,0.1948


## Per-class metrics

Accuracy hides what happens on the rare classes. Recall is the share of each class that is found, precision the share of the predictions of a class that are right.

In [4]:
for name, table in per_class.items():
    print(name)
    display(table.round(3))

salary


,precision,recall,f1,support
1,0.000,0.000,0.000,80
2,0.000,0.000,0.000,801
3,0.370,0.546,0.441,4000
4,0.318,0.113,0.167,3988
5,0.402,0.559,0.468,3986


no_salary


,precision,recall,f1,support
1,0.000,0.000,0.000,80
2,0.000,0.000,0.000,801
3,0.320,0.371,0.344,4000
4,0.311,0.311,0.311,3988
5,0.311,0.329,0.320,3986


In [5]:
pd.DataFrame({name: np.bincount(p, minlength=6)[1:] for name, p in preds.items()}, index=[1, 2, 3, 4, 5]).rename_axis('predicted class')

,salary,no_salary
predicted class,,
1,0,0
2,0,0
3,5899,4645
4,1422,3996
5,5534,4214
